In [ ]:
from syft_rds.orchestra import setup_rds_stack
from rds_chat_analysis import REPO_ROOT
import pandas as pd
import dotenv
from langchain.chat_models import init_chat_model
import os
from rds_chat_analysis import DATA_DIR

In [ ]:
key = "wildchat"
stack = setup_rds_stack(
    root_dir=REPO_ROOT / ".rds",
    key=key,
    log_level="DEBUG",
    reset=False,
)

do_client = stack.do_rds_client
ds_client = stack.ds_rds_client

In [ ]:
wildchat_dataset = ds_client.datasets[0]
wildchat_dataset.describe()

In [ ]:
private_data = pd.read_parquet(
    wildchat_dataset.private_path / "data.parquet",
)

private_data

# Load LLM

In [ ]:
dotenv.load_dotenv(wildchat_dataset.mock_path / "credentials.env")

llm = init_chat_model(
    model=os.environ["OPENROUTER_MODEL_NAME"],
    model_provider=os.environ["MODEL_PROVIDER"],
    openai_api_key=os.environ["OPENROUTER_API_KEY"],
    openai_api_base=os.environ["OPENROUTER_API_URL"],
)

In [ ]:
llm

# Extract facets


In [ ]:
from rds_chat_analysis.facets import extract_facets

cache_dir = DATA_DIR / "Wildchat-10k" / "cache"
cache_dir.mkdir(parents=True, exist_ok=True)

facet_df = extract_facets(
    conversation_df=private_data,
    llm=llm,
    llm_cache_dir=DATA_DIR / "Wildchat-10k" / "cache",
)

# Create embeddings

In [ ]:
from rds_chat_analysis.embeddings import embed_column, load_embedder

embedder = load_embedder(cache_dir=DATA_DIR / "Wildchat-10k" / "embedding_cache")

# filter rows where request is None
facet_df = facet_df[facet_df["request"].notna()]
facet_df = embed_column(
    facet_df,
    "request",
    embedder,
)

In [ ]:
import umap
import numpy as np
import pandas as pd
import hdbscan
from sklearn.metrics.pairwise import cosine_distances

request_embeddings = np.array(facet_df["request_embedding"].tolist())

# We use UMAP to R10 to reduce the dimensionality of the embeddings before clustering
# This is a common practice to improve clustering performance on high-dimensional data.
umap_for_clusterer = umap.UMAP(
    n_components=10,
    n_neighbors=15,
    min_dist=0.0,
    metric="cosine",
    random_state=42,
)

In [ ]:
# embeddings_for_clusterer = request_embeddings
embeddings_for_clusterer = umap_for_clusterer.fit_transform(request_embeddings)

# Precompute cosine distances for hdbscan
dists = cosine_distances(embeddings_for_clusterer).astype("float64")

# Infer the clustering parameters based on the size of the dataset
min_cluster_size = max(len(facet_df) // 500, 5)
min_samples = max(min_cluster_size // 3, 2)

hdbscan_model = hdbscan.HDBSCAN(
    min_cluster_size=min_cluster_size,
    min_samples=min_samples,
    metric="precomputed",
    cluster_selection_method="eom",
)
assignments = hdbscan_model.fit_predict(dists)
facet_df["request_cluster"] = assignments

print(
    f"Number of clusters found: {len(set(assignments)) - (1 if -1 in assignments else 0)}"
)
print(f"Ratio of unassigned points: {np.sum(assignments == -1) / len(assignments)}")

In [ ]:
umap_model = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.0,
    metric="cosine",
)

embedding_2d = umap_model.fit_transform(X=request_embeddings)
# embedding_2d = embeddings_for_clusterer

facet_df["embedding_2d"] = list(embedding_2d)

In [ ]:
import pandas as pd
import plotly.express as px

df = facet_df.copy()
df["x"] = df["embedding_2d"].apply(lambda v: v[0])
df["y"] = df["embedding_2d"].apply(lambda v: v[1])

fig = px.scatter(
    df,
    x="x",
    y="y",
    color=df["request_cluster"].astype(str),
    hover_data=["id"],
    title="Clustered Requests (Pre-description)",
    width=1200,
    height=800,
    color_discrete_sequence=px.colors.qualitative.Alphabet,
)

fig.update_layout(showlegend=False)
fig.show()

In [ ]:
from rds_chat_analysis.cluster_prompts import (
    format_cluster_description_prompt,
    FACET_CRITERIA,
)
from rds_chat_analysis.llm_utils import batch_process_llm_requests

clusters = facet_df["request_cluster"].unique()

cluster_description_responses = {}
prompts = []  # tuples (cluster_id, messages)
for cluster_id in clusters:
    if cluster_id == -1:
        continue

    in_df = facet_df[facet_df["request_cluster"] == cluster_id]
    out_df = facet_df[facet_df["request_cluster"] != cluster_id]

    in_sample = in_df["request"].sample(n=min(len(in_df), 50), random_state=42).tolist()
    out_sample = (
        out_df["request"].sample(n=min(len(out_df), 50), random_state=42).tolist()
    )

    messages = format_cluster_description_prompt(
        answers=in_sample,
        contrastive_answers=out_sample,
        criteria=FACET_CRITERIA["Request"],
    )

    prompts.append((cluster_id, messages))


messages_batch = [messages for _, messages in prompts]
ids_batch = [cluster_id for cluster_id, _ in prompts]
cluster_description_responses = batch_process_llm_requests(
    messages_batch,
    retrying_llm=llm.with_retry(),
    num_concurrent_requests=10,
)

In [ ]:
import re


def clean_description(text: str) -> dict[str, str]:
    if not text.startswith("<summary>"):
        text = "<summary>" + text
    summary = re.search(r"<summary>(.*?)</summary>", text, re.DOTALL)
    name = re.search(r"<name>(.*?)</name>", text, re.DOTALL)

    return {
        "name": name.group(1).strip() if name else "",
        "summary": summary.group(1).strip() if summary else "",
    }


cluster_descriptions = {
    -1: {
        "name": "Unassigned",
        "summary": "This cluster contains requests that do not fit into any other cluster.",
    },
}

for id_, response in zip(ids_batch, cluster_description_responses):
    cleaned = clean_description(response["content"])
    cluster_descriptions[int(id_)] = cleaned

In [ ]:
facet_df["cluster_name"] = facet_df["request_cluster"].map(
    lambda x: cluster_descriptions[x]["name"]
)
facet_df["cluster_summary"] = facet_df["request_cluster"].map(
    lambda x: cluster_descriptions[x]["summary"]
)

In [ ]:
publishable_fields = [
    "language",
    "embedding_2d",
    "request_cluster",
    "cluster_name",
    "cluster_summary",
    "turns",
]

publishable_df = facet_df[publishable_fields].copy()

In [ ]:
import plotly.express as px
import pandas as pd

embedding = pd.DataFrame(
    publishable_df["embedding_2d"].tolist(),
    columns=["x", "y"],
    index=publishable_df.index,
)
embedding = pd.concat([embedding, publishable_df.drop(columns="embedding_2d")], axis=1)

fig = px.scatter(
    embedding,
    x="x",
    y="y",
    color=embedding["request_cluster"].astype(str),
    hover_data={
        "cluster_name": True,
        "cluster_summary": False,
        "language": True,
        "turns": True,
        "x": False,
        "y": False,
    },
    title="Clustered Requests",
    width=1200,
    height=800,
    color_discrete_sequence=px.colors.qualitative.Alphabet,
)

fig.update_layout(showlegend=False)
fig.show()